# APIs d'IA dans le cloud : clés et points d'entrée

Un modèle servi par un fournisseur s'appelle par une requête HTTP, authentifiée par une clé d'API. Ce TP fait ce que montrent les slides « APIs sur étagère », « Points d'entrée » et « Clés d'API », sans compte de facturation :

1. Lire une clé dans les secrets de Colab ou de Codespaces
2. Appeler une API en HTTP brut
3. Faire le même appel par son SDK
4. Gérer les erreurs, les quotas et les délais
5. Révoquer et remplacer une clé

L'API principale est celle de Gemini (Google) : son palier gratuit ne demande pas de carte bancaire, elle est ouverte en France et dans l'UE, et dans l'Espace économique européen, la Suisse et le Royaume-Uni, Google applique au quota gratuit les conditions d'usage des données du palier payant (vos requêtes ne servent pas à améliorer ses produits ; [conditions](https://aistudio.google.com/docs/terms), octobre 2026).

Le notebook tourne sans GPU, sur Colab comme dans le Codespace. Sans clé, chaque cellule qui appelle une API le dit et explique quoi faire : « Exécuter tout » passe quand même.

## Préparation

**Une clé Gemini** : sur [Google AI Studio](https://aistudio.google.com/apikey), connectez-vous avec un compte Google, puis *Create API key*. Copiez-la, sans la coller dans ce notebook.

**La ranger dans les secrets**, sous le nom `GEMINI_API_KEY` :

- Colab : l'icône de clé (*Secrets*) dans la barre de gauche, *Ajouter un secret*, le nom et la valeur, puis activez *Accès au notebook*.
- Codespaces : sur GitHub, *Settings > Codespaces > Codespaces secrets > New secret*, le nom et la valeur, le dépôt `shuuchuu/labs-codespace` dans *Repository access*, puis arrêtez et redémarrez le Codespace : un secret n'arrive que dans un Codespace (re)démarré après lui, sous la forme d'une variable d'environnement.

## 1. Lire une clé dans les secrets

Écrivez `lit_secret(nom)`, qui renvoie la valeur du secret `nom`, ou `None` s'il n'y en a pas :

- dans Colab, avec `userdata.get` du module `google.colab`, en renvoyant `None` quand le secret n'existe pas, que le notebook n'y a pas accès ou que la demande n'aboutit pas ;
- ailleurs (le Codespace, votre machine), où `google.colab` n'existe pas, dans les variables d'environnement.

N'affichez jamais la valeur d'une clé : une sortie de cellule s'enregistre avec le notebook.

**Documentation utile**

- https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py
- https://docs.python.org/3/library/os.html#os.environ
- https://docs.github.com/fr/codespaces/managing-your-codespaces/managing-your-account-specific-secrets-for-github-codespaces

In [ ]:
def lit_secret(nom):
  """La valeur du secret `nom`, ou None s'il n'existe pas."""
  # Votre code ici
  ...

### Conseils supplémentaires

- `from google.colab import userdata` lève `ImportError` hors de Colab : c'est là que se fait le repli, dans un `try:` / `except ImportError:`
- `os.environ.get(nom)` : la variable d'environnement, ou `None` si elle n'existe pas
- `userdata.get(nom)` lève `userdata.SecretNotFoundError` (pas de secret de ce nom), `userdata.NotebookAccessError` (accès au notebook non activé) ou `userdata.TimeoutException` (pas de réponse de l'interface de Colab en 10 secondes)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import os


def lit_secret(nom):
  """La valeur du secret `nom`, ou None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab (Codespaces) : une variable d'environnement
    return ...
  try:
    return userdata.get(nom)
  except (userdata.SecretNotFoundError, ...):
    return None
```

### Solution

In [ ]:
import os


def lit_secret(nom):
  """La valeur du secret `nom`, ou None s'il n'existe pas."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab (Codespaces) : une variable d'environnement
    return os.environ.get(nom)
  try:
    return userdata.get(nom)
  except (
    userdata.SecretNotFoundError,
    userdata.NotebookAccessError,
    userdata.TimeoutException,
  ):
    return None

## 2. Appeler l'API en HTTP brut

L'API Gemini a un point d'entrée compatible avec celui d'OpenAI : la requête de la slide « Points d'entrée » y fonctionne telle quelle, avec l'URL, le modèle et la clé ci-dessous. Le modèle est une version fixée (`gemini-3.5-flash-lite`, stable, gratuite dans le palier gratuit), pas un alias comme `gemini-flash-latest`, que Google fait pointer vers chaque nouvelle version. Son prix au palier payant (octobre 2026, [tarifs](https://aistudio.google.com/docs/pricing)) : 0,30 $ par million de jetons en entrée, 2,50 $ en sortie.

In [ ]:
import httpx

URL_GEMINI = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODELE = "gemini-3.5-flash-lite"
# Prix du palier payant, en dollars par million de jetons (octobre 2026)
PRIX_ENTREE = 0.30
PRIX_SORTIE = 2.50
SANS_CLE_GEMINI = (
  "Pas de clé GEMINI_API_KEY : créez-la et rangez-la dans les secrets (voir"
  " « Préparation »), puis relancez la cellule qui la lit."
)

CLE_GEMINI = lit_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY :", "trouvée" if CLE_GEMINI else "absente")

Si la clé est là, envoyez avec `httpx.post` une question de votre choix au modèle `MODELE`, la clé dans l'en-tête `Authorization`, avec un délai maximal, et levez une exception si le code de retour est une erreur. Affichez le texte de la réponse, puis son champ `usage` : les jetons comptés, ce que le fournisseur facture (et ce que compte le quota). Sans clé, affichez `SANS_CLE_GEMINI`.

Calculez ensuite, avec `PRIX_ENTREE` et `PRIX_SORTIE`, ce que cet appel coûterait au palier payant, puis ce que coûteraient 100 000 appels par jour de cette taille.

En option : ajoutez à la requête un message `system` (par exemple « Réponds en une phrase. ») et `max_tokens`, et comparez les jetons comptés et le coût.

**Documentation utile**

- https://www.python-httpx.org/api/#helper-functions
- https://www.python-httpx.org/api/#response
- https://aistudio.google.com/docs/openai

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `httpx.post(url, headers=..., json=..., timeout=60)` : `json=` encode le dictionnaire et pose l'en-tête `Content-Type: application/json` ; sans `timeout`, httpx abandonne après 5 secondes, trop peu pour un LLM
- l'en-tête : `{"Authorization": f"Bearer {CLE_GEMINI}"}`
- le corps : `{"model": MODELE, "messages": [{"role": "user", "content": "..."}]}`
- `reponse.raise_for_status()` lève `httpx.HTTPStatusError` sur un code 4xx ou 5xx
- `reponse.json()["choices"][0]["message"]["content"]` : le texte ; `reponse.json()["usage"]` : les jetons de la question (`prompt_tokens`), de la réponse (`completion_tokens`) et le total (`total_tokens`)
- la sortie facturée compte aussi les jetons de raisonnement, que `completion_tokens` omet sur ce point d'entrée : elle vaut `total_tokens - prompt_tokens`. Le coût d'un appel : `(prompt_tokens * PRIX_ENTREE + (total_tokens - prompt_tokens) * PRIX_SORTIE) / 1_000_000`. Pour vérifier : 1 000 jetons en entrée et 500 en sortie coûtent 0,00155 $
- (option) le message `system` se place avant celui de l'utilisateur : `{"role": "system", "content": "..."}` ; `"max_tokens": ...` se place dans le corps, à côté de `"model"` ; trop bas, la réponse est coupée (`finish_reason` vaut `"length"`)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
if CLE_GEMINI:
  reponse = httpx.post(
    URL_GEMINI,
    headers={"Authorization": ...},
    json={
      "model": MODELE,
      "messages": [{"role": "user", "content": ...}],
    },
    timeout=60,
  )
  reponse.raise_for_status()
  corps = reponse.json()
  print(corps["choices"][0]["message"]["content"])
  usage = corps["usage"]
  print(usage)
  cout = (...) / 1_000_000
  print(f"Un appel : {cout:.6f} $")
  print(f"100 000 appels par jour : {100_000 * cout:.2f} $")
else:
  print(SANS_CLE_GEMINI)
```

### Solution

In [ ]:
if CLE_GEMINI:
  reponse = httpx.post(
    URL_GEMINI,
    headers={"Authorization": f"Bearer {CLE_GEMINI}"},
    json={
      "model": MODELE,
      "messages": [
        {"role": "user", "content": "En deux phrases : qu'est-ce qu'une clé d'API ?"}
      ],
    },
    timeout=60,
  )
  reponse.raise_for_status()
  corps = reponse.json()
  print(corps["choices"][0]["message"]["content"])
  usage = corps["usage"]
  print(usage)
  # sortie facturée : raisonnement compris, que completion_tokens omet
  sortie = usage["total_tokens"] - usage["prompt_tokens"]
  cout = (usage["prompt_tokens"] * PRIX_ENTREE + sortie * PRIX_SORTIE) / 1_000_000
  print(f"Un appel : {cout:.6f} $")
  print(f"100 000 appels par jour : {100_000 * cout:.2f} $")
else:
  print(SANS_CLE_GEMINI)

## 3. Le même appel par le SDK

Refaites la même question avec le SDK de Google, `google-genai` : un client créé avec la clé et un délai maximal, puis `generate_content`. Affichez le texte de la réponse et les jetons comptés. Comparez avec la partie 2 : qu'est-ce que le SDK fait pour vous, et qu'est-ce qu'il vous cache ?

**Documentation utile**

- https://aistudio.google.com/docs/text-generation
- https://googleapis.github.io/python-genai/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `genai.Client(api_key=CLE_GEMINI, http_options=types.HttpOptions(timeout=60_000))` : le délai du SDK est en **millisecondes** ; sans `api_key`, le client cherche les variables d'environnement `GEMINI_API_KEY` ou `GOOGLE_API_KEY` (pas les secrets de Colab, qui n'en sont pas)
- `client.models.generate_content(model=MODELE, contents="...")` renvoie une réponse typée : `reponse.text`, `reponse.usage_metadata`
- le SDK choisit l'URL, pose l'en-tête de la clé (`x-goog-api-key`, celui de l'API native de Gemini), encode et décode le JSON, et transforme une erreur HTTP en exception `google.genai.errors.APIError`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from google import genai
from google.genai import types

if CLE_GEMINI:
  client = genai.Client(api_key=CLE_GEMINI, http_options=types.HttpOptions(timeout=...))
  reponse = client.models.generate_content(model=MODELE, contents=...)
  print(reponse.text)
  print(...)
else:
  print(SANS_CLE_GEMINI)
```

### Solution

In [ ]:
from google import genai
from google.genai import types

if CLE_GEMINI:
  client = genai.Client(
    api_key=CLE_GEMINI, http_options=types.HttpOptions(timeout=60_000)
  )
  reponse = client.models.generate_content(
    model=MODELE, contents="En deux phrases : qu'est-ce qu'une clé d'API ?"
  )
  print(reponse.text)
  print(reponse.usage_metadata)
else:
  print(SANS_CLE_GEMINI)

**Ce que le SDK fait pour vous** : il choisit l'URL et l'en-tête de la clé (`x-goog-api-key`), encode et décode le JSON en objets typés (`reponse.text`, `usage_metadata`), transforme une erreur HTTP en `APIError`, et peut réessayer tout seul (option).

**Ce qu'il vous cache** : le format réel de la requête et de la réponse, et les codes HTTP (ils restent dans `erreur.code`).

## 4. Erreurs, quotas et délais

Une API tombe en erreur, et le code qui l'appelle doit savoir quand réessayer :

| Code | Cause | Que faire |
|---|---|---|
| 400, 401, 403 | Requête mal formée, clé absente, invalide ou révoquée, droits insuffisants | Ne pas réessayer : corriger la requête ou la clé |
| 429 | Quota ou limite de débit atteint (requêtes par minute, par jour…) | Attendre (l'en-tête `Retry-After`, quand il est là, dit combien de secondes), puis réessayer |
| 500, 502, 503, 504 | Panne ou surcharge du fournisseur | Réessayer, en attendant de plus en plus longtemps |
| Délai dépassé | Réseau lent, serveur surchargé | Réessayer, quelques fois au plus |

Les limites du palier gratuit de Gemini dépendent du modèle et du projet : Google AI Studio les affiche ([limites de débit](https://aistudio.google.com/docs/rate-limits)).

### 4.1 Réessayer quand il le faut

Écrivez `envoie_json(client, url, donnees, nb_essais=4, attente_base=1.0)`, qui envoie `donnees` en JSON avec le client httpx `client` (qui porte déjà la clé et le délai maximal) et renvoie la réponse réussie :

- après un 429, un 5xx ou un délai dépassé, elle attend puis réessaie, `nb_essais` fois en tout au plus ;
- l'attente vaut `Retry-After` quand la réponse le donne, sinon `attente_base`, puis le double à chaque nouvelle tentative (attente exponentielle) ;
- toute autre erreur, et l'erreur de la dernière tentative, lèvent leur exception (`httpx.HTTPStatusError`, `httpx.TimeoutException`).

Affichez chaque nouvel essai et son attente. Les cellules de la partie 4.2 essaient votre fonction.

**Documentation utile**

- https://www.python-httpx.org/api/#client
- https://www.python-httpx.org/exceptions/
- https://developer.mozilla.org/fr/docs/Web/HTTP/Reference/Headers/Retry-After

In [ ]:
def envoie_json(client, url, donnees, nb_essais=4, attente_base=1.0):
  """POST `donnees` en JSON, réessaie après un 429, un 5xx ou un délai dépassé.

  Renvoie la réponse réussie ; lève httpx.HTTPStatusError pour une autre erreur
  (requête ou clé refusée) et pour l'erreur de la dernière tentative.
  """
  # Votre code ici
  ...

#### Conseils supplémentaires

- `client.post(url, json=donnees)` lève `httpx.TimeoutException` quand le délai est dépassé
- `reponse.status_code` : le code ; `reponse.raise_for_status()` lève `httpx.HTTPStatusError` sur une erreur, et renvoie la réponse sinon
- `reponse.headers.get("Retry-After")` : `None` quand l'en-tête n'y est pas, une chaîne sinon (`"1"`)
- l'attente de la tentative `n` (à partir de 1) : `attente_base * 2 ** (n - 1)`, soit 1, 2, 4 secondes ; `time.sleep(attente)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import time

A_REESSAYER = {429, 500, 502, 503, 504}


def envoie_json(client, url, donnees, nb_essais=4, attente_base=1.0):
  """POST `donnees` en JSON, réessaie après un 429, un 5xx ou un délai dépassé."""
  for essai in range(1, nb_essais + 1):
    try:
      reponse = client.post(url, json=donnees)
    except httpx.TimeoutException:
      if essai == nb_essais:
        raise
      attente = ...
      print(f"  tentative {essai} : délai dépassé, nouvel essai dans {attente:.1f} s")
    else:
      if reponse.status_code not in A_REESSAYER or essai == nb_essais:
        return reponse.raise_for_status()
      attente = float(reponse.headers.get("Retry-After", ...))
      print(
        f"  tentative {essai} : {reponse.status_code},"
        f" nouvel essai dans {attente:.1f} s"
      )
    time.sleep(attente)
```

#### Solution

In [ ]:
import time

A_REESSAYER = {429, 500, 502, 503, 504}


def envoie_json(client, url, donnees, nb_essais=4, attente_base=1.0):
  """POST `donnees` en JSON, réessaie après un 429, un 5xx ou un délai dépassé.

  Renvoie la réponse réussie ; lève httpx.HTTPStatusError pour une autre erreur
  (requête ou clé refusée) et pour l'erreur de la dernière tentative.
  """
  for essai in range(1, nb_essais + 1):
    try:
      reponse = client.post(url, json=donnees)
    except httpx.TimeoutException:
      if essai == nb_essais:
        raise
      attente = attente_base * 2 ** (essai - 1)
      print(f"  tentative {essai} : délai dépassé, nouvel essai dans {attente:.1f} s")
    else:
      if reponse.status_code not in A_REESSAYER or essai == nb_essais:
        return reponse.raise_for_status()
      attente = float(
        reponse.headers.get("Retry-After", attente_base * 2 ** (essai - 1))
      )
      print(
        f"  tentative {essai} : {reponse.status_code},"
        f" nouvel essai dans {attente:.1f} s"
      )
    time.sleep(attente)

### 4.2 Essayer sur un faux serveur, puis sur le vrai

Provoquer un 429 sur la vraie API consommerait votre quota : `httpx.MockTransport` remplace le réseau par une fonction, qui rejoue ici une suite de réponses (un code HTTP, ou `"delai"` pour un délai dépassé). Pour chaque scénario, combien de requêtes votre fonction envoie-t-elle, et combien de temps attend-elle ?

In [ ]:
import time


def fausse_api(retours):
  """Un client httpx dont le « serveur » rejoue `retours`, un par requête."""
  appels = []

  def repond(requete):
    retour = retours[len(appels)]
    appels.append(retour)
    if retour == "delai":
      raise httpx.ReadTimeout("délai dépassé (simulé)", request=requete)
    if retour == 200:
      return httpx.Response(
        200, json={"choices": [{"message": {"content": "Bonjour"}}]}
      )
    entetes = {"Retry-After": "1"} if retour == 429 else {}
    return httpx.Response(retour, headers=entetes, json={"error": f"erreur {retour}"})

  return httpx.Client(transport=httpx.MockTransport(repond)), appels


scenarios = {
  "quota atteint deux fois": [429, 429, 200],
  "délai dépassé une fois": ["delai", 200],
  "panne passagère": [503, 503, 200],
  "clé invalide": [401],
  "quota épuisé": [429, 429, 429, 429],
}
for nom, retours in scenarios.items():
  print(f"{nom} : {retours}")
  client, appels = fausse_api(retours)
  debut = time.perf_counter()
  try:
    reponse = envoie_json(client, "https://api.example/v1/chat", {}, attente_base=0.5)
    issue = None if reponse is None else reponse.status_code
  except httpx.HTTPStatusError as erreur:
    issue = f"abandon, {erreur.response.status_code}"
  except httpx.TimeoutException:
    issue = "abandon, délai dépassé"
  duree = time.perf_counter() - debut
  print(f"  → {issue}, {len(appels)} requête(s) en {duree:.1f} s")

Les vraies APIs, avec une clé invalide (cela ne demande pas de compte). Gemini répond `400 INVALID_ARGUMENT`, pas `401` comme la plupart des APIs compatibles avec celle d'OpenAI : lisez le corps de l'erreur, pas seulement son code. Le point d'entrée compatible OpenAI n'en dit pas plus (son corps est une liste qui contient l'objet `error`) ; l'API native, celle du SDK, donne en plus la raison `API_KEY_INVALID` (dans `error.details`). Le SDK lève `google.genai.errors.ClientError` (4xx) ou `ServerError` (5xx), deux `APIError` ; il peut réessayer lui-même (`types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=4))`).

In [ ]:
from google import genai
from google.genai import errors

question = {"model": MODELE, "messages": [{"role": "user", "content": "Bonjour"}]}
with httpx.Client(headers={"Authorization": "Bearer cle-invalide"}, timeout=30) as http:
  try:
    envoie_json(http, URL_GEMINI, question)
  except httpx.HTTPStatusError as erreur:
    print(erreur.response.status_code, erreur.response.text[:200])
  except httpx.TransportError as erreur:
    print("Pas de réseau :", erreur)

client_invalide = genai.Client(api_key="cle-invalide")
try:
  client_invalide.models.generate_content(model=MODELE, contents="Bonjour")
except errors.APIError as erreur:
  print(type(erreur).__name__, erreur.code, erreur.status, erreur.message)
except httpx.TransportError as erreur:
  print("Pas de réseau :", erreur)

## 5. Révoquer et remplacer une clé

Supposons que `GEMINI_API_KEY` a fuité (collée dans une cellule, poussée sur GitHub). La cellule suivante en garde la valeur actuelle : exécutez-la **avant** de toucher à la clé.

In [ ]:
ancienne_cle = CLE_GEMINI

Écrivez `cle_valide(cle)`, qui renvoie `True` si Gemini accepte la clé, `False` sinon (en affichant le code de l'erreur). Essayez-la sur une clé inventée, puis faites la rotation :

1. sur la [page des clés de Google AI Studio](https://aistudio.google.com/apikey), créez une nouvelle clé, puis supprimez l'ancienne : elle ne marche plus nulle part ;
2. rangez la nouvelle valeur dans le secret `GEMINI_API_KEY` (Codespaces : puis redémarrez le Codespace, et réexécutez le notebook jusqu'à la cellule précédente) ;
3. relisez le secret, et vérifiez l'ancienne et la nouvelle clé.

**Documentation utile**

- https://aistudio.google.com/docs/api-key
- https://googleapis.github.io/python-genai/genai.html#genai.models.Models.get

In [ ]:
def cle_valide(cle):
  """True si Gemini accepte `cle`."""
  # Votre code ici
  ...

### Conseils supplémentaires

- `client.models.get(model=MODELE)` lit la fiche du modèle, sans consommer de jetons ; avec une clé refusée, il lève `google.genai.errors.ClientError`, dont `erreur.code` vaut 400
- gardez le client dans une variable : un `genai.Client(...)` qu'aucune variable ne retient est fermé avant d'envoyer sa requête
- une clé inventée, pour essayer : `"cle-inventee"`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from google import genai
from google.genai import errors


def cle_valide(cle):
  """True si Gemini accepte `cle`."""
  client = genai.Client(api_key=...)
  try:
    client.models.get(model=MODELE)
  except errors.ClientError as erreur:
    print("  refusée :", erreur.code)
    return False
  return True


print("clé inventée :", cle_valide("cle-inventee"))
if ancienne_cle:
  CLE_GEMINI = lit_secret("GEMINI_API_KEY")
  print("ancienne clé :", cle_valide(...))
  print("nouvelle clé :", cle_valide(...))
```

### Solution

In [ ]:
from google import genai
from google.genai import errors


def cle_valide(cle):
  """True si Gemini accepte `cle`."""
  client = genai.Client(api_key=cle)
  try:
    client.models.get(model=MODELE)
  except errors.ClientError as erreur:
    print("  refusée :", erreur.code)
    return False
  return True


print("clé inventée :", cle_valide("cle-inventee"))
if ancienne_cle:
  CLE_GEMINI = lit_secret("GEMINI_API_KEY")
  print("ancienne clé :", cle_valide(ancienne_cle))
  print("nouvelle clé :", cle_valide(CLE_GEMINI))
else:
  print(SANS_CLE_GEMINI)

## 6. Faire le ménage

Supprimez les clés dont vous n'avez plus besoin, dans Google AI Studio. Une clé par usage, et pas une de plus.